# Tutorial 1.5: Prompt Management

![](images/6_Prompt-Management.png)

## Version Control for Prompts with the Prompt Registry

Welcome to prompt management! Prompts are the instructions that guide LLMs, and managing them properly is crucial for maintaining quality and collaboration in GenAI applications.

![mlflow_arch](./images/mlflow_gen_arch.png)

### What You'll Learn
- Why the Prompt Registry solves prompt management problems
- Registering prompts with `mlflow.genai.register_prompt()`
- Using Jinja2 `{{ variable }}` templates
- Versioning prompts with commit messages
- Using the Registry as a shared team library
- Aliases (`@production`, `@staging`) for safe deployments
- Searching prompts across your organization

### Prerequisites
- Completed previous notebooks (1.1-1.4)
- Understanding of experiment tracking

### Estimated Time: 15-20 minutes

---
## Step 1: Why Manage Prompts?

### The Problem

Without proper management:

```python
# Scattered prompts in code
prompt_assistant = "You are a helpful assistant. Answer: {question}"
prompt_repond = "You are helpful. Respond to: {question}"  # Which one works better?
prompt_qa = "Answer {question}"  # Lost track of what works
```

**Problems:**
- ❌ Prompts hardcoded in multiple places
- ❌ No version history
- ❌ Hard to A/B test different versions
- ❌ Difficult to collaborate
- ❌ Can't track which prompt generated which output

### The Solution: MLflow Prompt Registry

```python
# Centralized, versioned prompts in the Prompt Registry
prompt = mlflow.genai.register_prompt(
    name="my-qa-prompt",
    template="You are a helpful assistant. Answer: {{ question }}",
    commit_message="Initial version"
)
# Now you can:
# ✅ Version prompts automatically
# ✅ Load by name, version, or alias
# ✅ Share with team
# ✅ Search across your organization
```

### Benefits

1. **Reproducibility**: Know exactly which prompt was used
2. **Collaboration**: Share prompts across team
3. **Experimentation**: Systematic A/B testing
4. **Version Control**: Track prompt evolution with commit messages
5. **Governance**: Audit and approval processes via aliases

---
## Step 2: Environment Setup

In [ ]:
import mlflow
from dotenv import load_dotenv
from utils.clnt_utils import is_databricks_ai_gateway_client, get_databricks_ai_gateway_client, get_openai_client, get_ai_gateway_model_names

# Load environment
load_dotenv()

# Configure MLflow
mlflow.set_tracking_uri("http://localhost:5000")

use_databricks_provider = is_databricks_ai_gateway_client()
if use_databricks_provider:
    client = get_databricks_ai_gateway_client()
    model_name = get_ai_gateway_model_names()[0]
else:
    client = get_openai_client()
    model_name = "gpt-4o-mini"

# Create experiment and enable autologging
mlflow.set_experiment("05-prompt-management")
mlflow.openai.autolog()

provider = "Databricks AI Gateway" if use_databricks_provider else "OpenAI"
print(f"✅ MLflow experiment '05-prompt-management' ready | Provider: {provider}")
print(f"   Tracking URI: {mlflow.get_tracking_uri()} | Autologging: ENABLED")

---
## Step 3: Your First Prompt in the Registry

The **Prompt Registry** is MLflow's centralized store for managing, versioning, and sharing prompts across your team and applications.

> **Note:** The Registry uses **Jinja2 syntax** `{{ variable }}` for template variables instead of Python's `{variable}` format strings.

In [ ]:
import mlflow

print("📝 Registering your first prompt to the Prompt Registry...\n")

# Register a simple QA prompt — note Jinja2 {{ variable }} syntax
prompt = mlflow.genai.register_prompt(
    name="tutorial-first-qa",
    template="""You are a helpful AI assistant.

Answer the following question concisely:
{{ question }}

Answer:""",
    commit_message="Initial version - general purpose QA",
    tags={
        "author": "jules",
        "use_case": "general_qa",
    }
)

print(f"✅ Registered: {prompt.name} (version {prompt.version})")
print(f"   Commit: {prompt.commit_message}")

# Load it back by name (gets latest version)
loaded = mlflow.genai.load_prompt("tutorial-first-qa")
print(f"\n📥 Loaded prompt (version {loaded.version})")

# Fill the template with .format()
question = "What is prompt engineering?"
prompt_filled = loaded.format(question=question)

# Call the LLM
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": prompt_filled}],
)

answer = response.choices[0].message.content
print(f"\n❓ Question: {question}")
print(f"🤖 Answer: {answer}")
print("\n✅ Your first Registry prompt is live!")
print("   View it: MLflow UI → Prompt Registry (left sidebar)")

---
## Step 4: Prompt Templates with Variables

Real-world prompts need dynamic content — roles, context, tasks. Register multi-variable templates directly in the Registry using **Jinja2 `{{ variable }}` syntax**.

The Prompt object's `.format()` method fills all variables at once, just like Python's `.format()` but for Jinja2 templates.

### Register a multi-variable role-based template

In [ ]:
# Register a multi-variable role-based prompt directly in the Registry
role_prompt = mlflow.genai.register_prompt(
    name="tutorial-role-based-qa",
    template="""You are a {{ role }}.

Context: {{ context }}

Task: {{ task }}

Provide your response:""",
    commit_message="Multi-variable role-based template",
    tags={
        "author": "jules",
        "variables": "role,context,task",
        "use_case": "role_based_qa"
    }
)

print(f"✅ Registered: {role_prompt.name} (version {role_prompt.version})")

# Load and fill the template
loaded_role_prompt = mlflow.genai.load_prompt("tutorial-role-based-qa")

prompt_text = loaded_role_prompt.format(
    role="technical documentation expert",
    context="MLflow is an open source AI platform",
    task="Explain MLflow tracing in 2 sentences"
)

print(f"\n📝 Filled prompt:\n{prompt_text}")

response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": prompt_text}],
)

answer = response.choices[0].message.content
print(f"\n🤖 Answer: {answer}")
print(f"\n📋 Variables: role, context, task  |  Version: {loaded_role_prompt.version}")

---
## Step 5: Versioning Prompts in the Registry

Call `register_prompt()` on the **same name** multiple times to create new versions automatically. Each version gets a `commit_message` documenting what changed — no `mlflow.log_param("prompt_version", ...)` needed.

> **Note:** If you've run this notebook before, version numbers will continue from where they left off. That's normal — the Registry tracks the full history.

In [ ]:
question = "What is MLflow?"
print("🔄 Creating prompt versions via the Registry...\n")

# Version 1: Basic — minimal prompt, no guidance
p_v1 = mlflow.genai.register_prompt(
    name="tutorial-versioned-qa",
    template="Answer this question: {{ question }}",
    commit_message="Initial basic version"
)
print(f"✅ Created version {p_v1.version}: Basic (no guidance)")

# Version 2: Full structured format with role + guidelines
p_v2 = mlflow.genai.register_prompt(
    name="tutorial-versioned-qa",
    template="""You are a helpful AI assistant specializing in technical topics.

Guidelines:
  - Answer concisely (2-3 sentences max)
  - Be accurate and factual
  - If unsure, say so

Question: {{ question }}

Answer:""",
    commit_message="Added specialization, role, and explicit guidelines"
)
print(f"✅ Created version {p_v2.version}: With role + guidelines")

# Load and compare both versions
print(f"\n🔍 Comparing versions for: '{question}'\n")
for ver in [p_v1.version, p_v2.version]:
    p = mlflow.genai.load_prompt(f"prompts:/tutorial-versioned-qa/{ver}")
    filled = p.format(question=question)
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": filled}],
        temperature=1.0
    )
    answer = response.choices[0].message.content
    print(f"v{ver}: {answer[:100]}...")

print("\n✅ Both versions tracked in the Registry!")
print("   View history: MLflow UI → Prompt Registry → tutorial-versioned-qa")

### 📈 What You Get with Registry Versioning

| Feature | What it means |
|---------|---------------|
| Automatic version numbers | No manual tracking needed |
| `commit_message` | Documents *why* each change was made |
| Load by version | `load_prompt("prompts:/name/2")` to pin to a specific version |
| Full history in UI | Prompt Registry → click a prompt → see all versions |

**In production**, use aliases (`@production`) rather than version numbers so deployments don't need code changes when you update a prompt.

---
## Step 6: Linking Prompts to Experiments

Knowing *which prompt produced which result* is essential for reproducibility and debugging. When you load a prompt from the Registry and use it in a run, log its **name, version, and URI** as run params — then you can always reload the exact prompt that generated any output.

```python
# Later, reproduce any run exactly:
prompt = mlflow.genai.load_prompt(run.data.params["prompt_uri"])
```

> **Why `prompt.uri`?** The URI (e.g. `prompts:/tutorial-versioned-qa/2`) is a pinned, immutable reference. Unlike a name alone, it survives future versions and alias changes.

In [ ]:
print("🔗 Linking a Registry prompt to an experiment run...\n")

# Load the versioned prompt you want to use
prompt = mlflow.genai.load_prompt("prompts:/tutorial-versioned-qa/2")

with mlflow.start_run(run_name="qa-with-linked-prompt"):  # I can also experiment_id=id as a parameter
    question = "What is MLflow tracing?"
    filled = prompt.format(question=question)

    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": filled}],
    )
    answer = response.choices[0].message.content

    # Log prompt identity, input, and output together as a single artifact
    mlflow.log_dict({
        "prompt_name":    prompt.name,
        "prompt_version": prompt.version,
        "prompt_uri":     prompt.uri,
        "question":       question,
        "answer":         answer,
    }, "run_record.json")

    run_id = mlflow.active_run().info.run_id
    print(f"✅ Run logged: {run_id}")
    print(f"   prompt_name:    {prompt.name}")
    print(f"   prompt_version: {prompt.version}")
    print(f"   prompt_uri:     {prompt.uri}")
    print(f"\n🤖 Answer: {answer}")

print("\n💡 View in MLflow UI → Experiments → qa-with-linked-prompt → Artifacts → run_record.json")
print("   Reproduce any run: mlflow.genai.load_prompt(run_record['prompt_uri'])")

---
## Step 7: Prompt Registry as a Shared Library

The Prompt Registry serves as your team's centralized prompt library. Instead of a Python dict living in one codebase, prompts in the Registry are:

- **Discoverable** via `search_prompts()` — anyone can find them
- **Loadable** from any application with `load_prompt()`
- **Versioned** with full history and commit messages
- **Shareable** across teams without code sharing

In [ ]:
# Define library prompts using Jinja2 {{ var }} syntax — ready to register directly
PROMPT_LIBRARY = {
    "qa_simple": {
        "template": "Answer this question: {{ question }}",
        "variables": ["question"],
        "use_case": "Simple Q&A",
    },
    "qa_with_context": {
        "template": """Use the following context to answer the question.
Context: {{ context }}

Question: {{ question }}

Answer based only on the context above:""",
        "variables": ["context", "question"],
        "use_case": "RAG Q&A",
    },
    "classification": {
        "template": "Classify the following text into one of these categories: {{ categories }}\n\nText: {{ text }}\n\nCategory:",
        "variables": ["text", "categories"],
        "use_case": "Text classification",
    },
}

print("📚 Library prompts defined:")
for name, info in PROMPT_LIBRARY.items():
    print(f"   - {name}: {info['use_case']}")
print("\n   Next: register all to the Prompt Registry →")

### Register the Library to the Prompt Registry

Register all library prompts with `register_prompt()`. The Registry becomes your team's centralized, searchable prompt store — accessible from any application.

In [ ]:
print("🔄 Registering library prompts to the Prompt Registry...\n")

registered_prompts = {}
for name, prompt_info in PROMPT_LIBRARY.items():
    # prefix with tutorial to avoid conflicts with other prompts
    prompt_name = f"tutorial-{name}"

    prompt = mlflow.genai.register_prompt(
        name=prompt_name,
        template=prompt_info["template"],
        commit_message=f"Initial version - {prompt_info['use_case']}",
        tags={
            "use_case": prompt_info["use_case"],
            "variables": ",".join(prompt_info["variables"]),
            "author": "jules"
        }
    )

    registered_prompts[name] = prompt
    print(f"✅ Registered: {prompt_name} (version {prompt.version})")

print(f"\n📋 {len(registered_prompts)} prompts registered to the Prompt Registry")
print("   Browse: MLflow UI → Prompt Registry")

### Use Library Prompts via `load_prompt()`

With prompts in the Registry, any application loads them by name — no Python dict or shared code needed.

In [ ]:
# Load and use prompts from the Registry — no Python dict needed
print("📚 Using prompts from the Registry...\n")

# Example 1: RAG Q&A
rag_prompt = mlflow.genai.load_prompt("tutorial-qa_with_context")
filled = rag_prompt.format(
    context="MLflow Tracing provides observability for GenAI applications.",
    question="What does MLflow Tracing do?"
)
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": filled}],
)
print(f"RAG Q&A: {response.choices[0].message.content}\n")

# Example 2: Classification
cls_prompt = mlflow.genai.load_prompt("tutorial-classification")
filled = cls_prompt.format(
    text="I love using MLflow for my ML projects!",
    categories="Positive, Negative, Neutral"
)
response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": filled}],
)
print(f"Classification: {response.choices[0].message.content}")

print("\n✅ Registry prompts are accessible from any app with MLflow access!")
print("   Search available prompts with: mlflow.genai.search_prompts()")

---
## Step 8: Aliases for Environment Management

Aliases let production code reference prompts by **environment name** (`@production`, `@staging`) rather than version numbers. This enables:

- **Safe rollouts**: test new versions in staging before promoting to production
- **One-click rollback**: just update the alias to the previous version
- **Zero application code changes**: your app always loads `@production`

```python
# Application code never changes — only the alias target changes
prompt = mlflow.genai.load_prompt("prompts:/tutorial-qa_with_context@production")
```

In [ ]:
#
# Set up aliases for the RAG Q&A prompt
print("🏷️ Setting up aliases for environment management...\n")

# Set version 1 as production — the current stable version serving users
mlflow.genai.set_prompt_alias("tutorial-qa_with_context", "production", version=1)
print("✅ Set 'production' alias → version 1")

# Staging also starts at version 1 — we'll advance it to a new version in the next step
mlflow.genai.set_prompt_alias("tutorial-qa_with_context", "staging", version=1)
print("✅ Set 'staging' alias → version 1")

# Load prompt via alias - this is what production code should use!
print("\n📥 Loading prompt via alias...")
prod_prompt = mlflow.genai.load_prompt("prompts:/tutorial-qa_with_context@production")
print(f"   Loaded production prompt (version {prod_prompt.version})")

# Use the production prompt
filled = prod_prompt.format(
    context="MLflow is an open source platform for the ML lifecycle.",
    question="What is MLflow?"
)

response = client.chat.completions.create(
    model=model_name,
    messages=[{"role": "user", "content": filled}],
)

print(f"\n🤖 Production response: {response.choices[0].message.content}")
print("\n💡 Next: register an improved version and advance staging to it →")

### Versioning with Commit Messages

Register improved versions with descriptive commit messages to track changes.

In [ ]:
# Register an improved version of the RAG Q&A prompt
print("📝 Registering an improved version...\n")

improved_template = """You are a helpful AI assistant. Use ONLY the provided context to answer.

Context:
{{ context }}

Question: {{ question }}

Instructions:
- Answer based strictly on the context above
- If the answer is not in the context, say "I don't have enough information"
- Be concise (2-3 sentences max)

Answer:"""

# Register as a new version — same name, new commit_message
updated_prompt = mlflow.genai.register_prompt(
    name="tutorial-qa_with_context",
    template=improved_template,
    commit_message="Added strict context adherence instructions and conciseness requirement",
    tags={
        "use_case": "RAG Q&A",
        "variables": "context,question",
        "author": "jules"
    }
)

print(f"✅ Registered version {updated_prompt.version}")
print("   Commit: Added strict context adherence instructions")

# Promote to staging — production still points to previous version
mlflow.genai.set_prompt_alias("tutorial-qa_with_context", "staging", version=updated_prompt.version)
print(f"\n🏷️  Promoted version {updated_prompt.version} → 'staging'")
print("   Production still points to previous version (safe rollout!)")
print("\n💡 When staging looks good:")
print("   mlflow.genai.set_prompt_alias('tutorial-qa_with_context', 'production', version=updated_prompt.version)")

### Searching Prompts

Discover prompts across your organization using tag-based search.

In [ ]:
# Search for prompts by tags
print("🔍 Searching prompts in the Registry...\n")

# Search all prompts by author
all_prompts = mlflow.genai.search_prompts(filter_string="tags.author='jules'")
print(f"Found {len(all_prompts)} prompts by author 'jules':")
for p in all_prompts:
    print(f"   - {p.name}")

# Filter by use_case
rag_prompts = mlflow.genai.search_prompts(filter_string="tags.use_case='RAG Q&A'")
print(f"\nFound {len(rag_prompts)} RAG Q&A prompts:")
for p in rag_prompts:
    print(f"   - {p.name}")

print("\n💡 Search tips:")
print("   - Filter by author:   tags.author='jules'")
print("   - Filter by use case: tags.use_case='RAG Q&A'")
print("   - Combine:            tags.author='jules' AND tags.use_case='RAG Q&A'")

---
## Step 9: Best Practices

### 📝 Creation

- **Use descriptive names** — `rag_qa_with_context` not `prompt1`
- **Use tags generously** — `author`, `use_case`, `team` power `search_prompts()`
- **Use Jinja2 `{{ descriptive_names }}`** and list variables in a `"variables"` tag
- **Write meaningful commit messages** — `"Added strict context adherence"` not `"update"`

### 🔄 Versioning

- **Version = call `register_prompt()` again** on the same name — the Registry auto-assigns the number
- **Always include `commit_message`** documenting *why* the change was made
- **Use aliases in production** — never hardcode version numbers
  - ✅ `load_prompt("prompts:/my-prompt@production")`
  - ❌ `load_prompt("prompts:/my-prompt/3")`

### 🚀 Deployment via Aliases

- **Gradual rollout:** new version → `staging` alias → monitor → promote to `production` alias
- **Rollback = update alias** back to the previous version — no code changes needed
- **Application code never changes:** `mlflow.genai.load_prompt("prompts:/my-prompt@production")`

### 👥 Collaboration

- **Centralize in the Registry** — one source of truth for the whole team
- **Tag consistently** — agree on a taxonomy (`author`, `use_case`, `team`, `status`)
- **Search before creating** — `search_prompts()` to avoid duplicating existing prompts

---
## Summary

In this notebook, you learned to manage prompts using the **MLflow Prompt Registry** — your team's single source of truth for prompts.

1. ✅ Why the Prompt Registry solves prompt management problems
2. ✅ **Register your first prompt** with `mlflow.genai.register_prompt()`
3. ✅ **Multi-variable templates** using Jinja2 `{{ variable }}` syntax
4. ✅ **Version prompts** by calling `register_prompt()` with `commit_message=`
5. ✅ **Link prompts to experiment runs** — log `prompt_name`, `prompt_version`, `prompt_uri` as run params
6. ✅ **Prompt Registry as a shared library** — register once, load from anywhere
7. ✅ **Aliases** (`@production`, `@staging`) for safe deployments and easy rollback
8. ✅ **Promote new versions** from staging to production via alias updates
9. ✅ **Search prompts** by tag across your organization
10. ✅ Best practices for team collaboration

### Key Takeaways

- **Prompt Registry is your single source of truth** — no more prompts scattered in code
- **Use Jinja2 syntax** `{{ variable }}` in Registry templates
- **Version everything** — commit messages tell you *why* each change was made
- **Link prompts to runs** — log `prompt.uri` so every result is reproducible
- **Use aliases in production** — `@production` decouples deployments from version numbers
- **Search to discover** — `search_prompts(filter_string="tags.use_case='RAG Q&A'")` finds what you need

### What's Next?

**📓 Notebook 1.6: Framework Integrations**

Learn how to:
- Work with different GenAI frameworks
- Compare OpenAI, LangChain, LlamaIndex
- Choose the right tool for your use case
- Integrate MLflow with your framework
- Best practices for each framework